# 06 RAG 知识库工程


## 1 本章要完成什么

本章只使用一个场景：为“萤火协作平台”的五份内部文档做问答助手。

用户最终可以问：`萤火平台支持哪些部署方式？` 系统先找到产品说明中的证据，再把证据交给生成模型，并返回可核验的来源。模型参数中没有这些内部文档，文档更新后也不需要重新训练模型。

```text
五份 Markdown 文档
  -> 解析与切分
  -> 建立关键词索引和向量索引
  -> 检索候选
  -> 权限过滤、融合与 Rerank
  -> 组装带来源的 Context
  -> 生成答案或拒答
```

我们先做一个能工作的关键词版本，再用它暴露同义表达、Chunk 边界和排序问题。Embedding、Hybrid、Rerank、ACL 和无答案检测都用于修复一个已经看到的失败。最后再把教学版本扩展为可更新、可恢复的生产架构。


## 2 RAG 是什么

RAG（Retrieval-Augmented Generation，检索增强生成）在模型生成答案前，从外部知识库取回相关证据，并把证据放入本次请求的 Context。[1]

它包含两个动作：

1. **Retrieval**：根据问题找到可能包含答案的 Chunk。
2. **Generation**：让生成模型只根据这些 Chunk 组织答案并标注来源。

因此，RAG 的质量也要拆成两部分检查：检索是否找对证据，生成是否忠实使用证据。向量数据库只参与检索，不能保证最终答案正确。

RAG 适合内部文档、持续更新的知识和需要引用的问答。长上下文适合一次性读取少量已知材料；微调适合改变稳定行为或格式；Tool Calling 适合查询结构化系统、计算或执行动作。


### 2.1 先建立全局地图

```mermaid
flowchart LR
  subgraph Indexing["索引链路：提前处理"]
    D["Document"] --> P["解析"] --> C["Chunk + Metadata"] --> I["关键词/向量索引"]
  end
  subgraph Query["查询链路：每次提问"]
    Q["问题"] --> R["检索"] --> F["ACL / 融合 / Rerank"] --> X["Context"] --> L["生成或拒答"]
  end
  I --> R
```

`Document` 是原始文件；`Chunk` 是参与检索的文本片段；`Metadata` 保存文档 ID、版本和权限；`Embedding` 是模型输出的定长向量；`Retriever` 取候选；`Reranker` 对候选二次排序。后面每出现一个术语，都会立即看到对应的数据。


## 3 第一个可运行版本：关键词 RAG

目标问题是 `萤火平台支持哪些部署方式？`。第一版只做四步：加载文档、切成 Chunk、用 BM25 检索、组装 Context。先跑通闭环，再观察它为什么不够。


In [1]:
import os
import sys
from pathlib import Path

# 检测仓库根目录
def _find_repo_root():
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "chapters" / "06-rag").is_dir():
            return start
    return Path.cwd()

REPO = _find_repo_root()
os.chdir(REPO)

DOCS_DIR = REPO / "chapters" / "06-rag" / "examples" / "docs"
doc_files = sorted(DOCS_DIR.glob("*.md"))
for f in doc_files:
    print(f"{f.stem}: {len(f.read_text())} chars")
print(f"\nTotal: {len(doc_files)} documents")


doc-001-product-overview: 605 chars
doc-002-api-reference: 1493 chars
doc-003-changelog: 694 chars
doc-004-incident-handbook: 1024 chars
doc-005-team-conventions: 965 chars

Total: 5 documents


In [2]:
import re

def load_document(path: str) -> dict:
    text = Path(path).read_text(encoding="utf-8")
    # 提取标题（第一个 # 开头的行）
    title_match = re.search(r"^#\s+(.+)", text, re.MULTILINE)
    title = title_match.group(1) if title_match else Path(path).stem
    return {"text": text, "title": title, "path": path}

# 加载五份文档
docs = {f.stem: load_document(str(f)) for f in doc_files}
for doc_id, doc in docs.items():
    print(f"{doc_id}: title='{doc['title']}', {len(doc['text'])} chars")


doc-001-product-overview: title='萤火协作平台产品说明', 605 chars
doc-002-api-reference: title='萤火协作平台 API 参考', 1493 chars
doc-003-changelog: title='萤火协作平台发布记录', 694 chars
doc-004-incident-handbook: title='萤火协作平台故障处理手册', 1024 chars
doc-005-team-conventions: title='萤火协作平台团队规范', 965 chars


### 3.1 文档加载后得到什么

上一个 Cell 把磁盘文件变成 `docs` 字典。每个元素仍是一整份文档：

```python
{"title": "萤火协作平台产品说明", "text": "完整正文...", "path": "来源路径"}
```

整份文档通常太长，检索结果也不够精确。下一步把正文切成较小的 Chunk，并保留来源字段。


In [3]:
# 第一版先按固定字符切分。后面会专门制造边界问题并改进它。
def chunk_fixed_size(text: str, size: int = 256, overlap: int = 50) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap  # 下一块向前回退 50 字，保留边界上下文
    return chunks

baseline_chunks = []
for doc_id, doc in docs.items():
    for chunk_index, text in enumerate(chunk_fixed_size(doc["text"])):
        baseline_chunks.append({"chunk_id": f"{doc_id}:chunk_{chunk_index}",
            "doc_id": doc_id, "title": doc["title"], "text": text,
            "access_level": "ops-only" if "ops-only" in doc["text"].lower() else "public"})

print("文档数：", len(docs), "Chunk 数：", len(baseline_chunks))
print("第一条 Chunk：", baseline_chunks[0])


文档数： 5 Chunk 数： 25
第一条 Chunk： {'chunk_id': 'doc-001-product-overview:chunk_0', 'doc_id': 'doc-001-product-overview', 'title': '萤火协作平台产品说明', 'text': '# 萤火协作平台产品说明\n\n## 产品定位\n\n萤火协作平台是一套面向中小型研发团队的协作工具集，提供任务管理、文档共享、即时通讯和自动化流水线四个核心模块。\n\n## 用户角色\n\n- **管理员**：管理团队、配置权限、查看审计日志\n- **项目负责人**：创建项目、分配任务、审批发布\n- **研发工程师**：查看和更新任务、提交代码、参与代码审查\n- **测试工程师**：创建测试用例、提交缺陷、验证修复\n- **产品经理**：管理需求、排定优先级、查看进度报告\n\n## 部署方式\n\n平台支持以下部署方式：\n\n', 'access_level': 'public'}


### 3.2 BM25 怎样给关键词结果排序

BM25 根据查询词在某个 Chunk 中出现的次数、这个词在全库中的稀有程度，以及 Chunk 长度计算分数。常用形式为：[3]

$$
score(D,Q)=\sum_{q\in Q} IDF(q)\cdot
\frac{f(q,D)(k_1+1)}{f(q,D)+k_1(1-b+b\frac{|D|}{avgdl})}
$$

- $f(q,D)$：查询词 $q$ 在 Chunk $D$ 中出现多少次。
- $IDF(q)$：词越少见，区分能力越强，权重越高。
- $|D|/avgdl$：长度修正，避免长 Chunk 仅靠词多获得优势。
- $k_1$ 和 $b$：控制词频饱和与长度修正。

读代码时只跟踪三份数据：`tokenized_chunks` 是建库输入，`scores` 与 Chunk 一一对应，排序后的下标用来取回原文。


In [4]:
from rank_bm25 import BM25Okapi
import jieba
jieba.setLogLevel(jieba.logging.WARN)  # 隐藏词典加载日志，只保留实验输出

def tokenize(text: str) -> list[str]:
    """中文没有天然空格，先用 jieba 把句子拆成词。"""
    return [word.strip() for word in jieba.cut(text) if word.strip()]

# 索引阶段：每个 Chunk 的 token 列表占一行。
tokenized_chunks = [tokenize(chunk["text"]) for chunk in baseline_chunks]
bm25 = BM25Okapi(tokenized_chunks)

def keyword_search(query: str, top_k: int = 3) -> list[dict]:
    query_terms = tokenize(query)
    scores = bm25.get_scores(query_terms)  # 第 i 个分数对应第 i 个 Chunk
    ranked_ids = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [{"chunk": baseline_chunks[i], "score": float(scores[i])}
            for i in ranked_ids[:top_k]]

question = "萤火平台支持哪些部署方式？"
keyword_results = keyword_search(question, top_k=2)
print("查询分词：", tokenize(question))
for rank, item in enumerate(keyword_results, 1):
    print(f"#{rank} score={item['score']:.3f} {item['chunk']['chunk_id']}")
    print(item['chunk']['text'][:140].replace("\n", " "))


查询分词： ['萤火', '平台', '支持', '哪些', '部署', '方式', '？']
#1 score=9.437 doc-001-product-overview:chunk_1
**产品经理**：管理需求、排定优先级、查看进度报告  ## 部署方式  平台支持以下部署方式：  1. **SaaS 云端版**：由萤火团队托管，自动升级，无需维护服务器。 2. **私有部署版**：部署在客户的云服务器或本地机房。最低要求 4 核 CPU、8 GB 内存、50
#2 score=8.318 doc-001-product-overview:chunk_0
# 萤火协作平台产品说明  ## 产品定位  萤火协作平台是一套面向中小型研发团队的协作工具集，提供任务管理、文档共享、即时通讯和自动化流水线四个核心模块。  ## 用户角色  - **管理员**：管理团队、配置权限、查看审计日志 - **项目负责人**：创建项目、分配任务、审批


### 3.3 检索结果怎样进入模型

检索器返回的是 Chunk，生成模型需要一段明确区分“资料”和“问题”的 Prompt。来源编号必须与 Chunk ID 一起保存，不能只在答案末尾拼一个 URL。


In [5]:
def build_context(results: list[dict]) -> str:
    return "\n\n---\n\n".join(
        f"[来源 {rank}] {item['chunk']['title']} | {item['chunk']['chunk_id']}\n"
        f"{item['chunk']['text']}"
        for rank, item in enumerate(results, 1))

context = build_context(keyword_results)
prompt = f"""只根据资料回答问题；资料不足时说明无法回答。
资料：
{context}

问题：{question}
请在相关句后标注 [来源 n]。"""
print(prompt[:1200])


只根据资料回答问题；资料不足时说明无法回答。
资料：
[来源 1] 萤火协作平台产品说明 | doc-001-product-overview:chunk_1
**产品经理**：管理需求、排定优先级、查看进度报告

## 部署方式

平台支持以下部署方式：

1. **SaaS 云端版**：由萤火团队托管，自动升级，无需维护服务器。
2. **私有部署版**：部署在客户的云服务器或本地机房。最低要求 4 核 CPU、8 GB 内存、50 GB 可用磁盘。
3. **混合部署版**：核心服务部署在客户侧，文件存储和通知服务使用萤火云端。

Docker 一键部署命令：

```bash
docker run -d -p 8080:8080 firefly/platf

---

[来源 2] 萤火协作平台产品说明 | doc-001-product-overview:chunk_0
# 萤火协作平台产品说明

## 产品定位

萤火协作平台是一套面向中小型研发团队的协作工具集，提供任务管理、文档共享、即时通讯和自动化流水线四个核心模块。

## 用户角色

- **管理员**：管理团队、配置权限、查看审计日志
- **项目负责人**：创建项目、分配任务、审批发布
- **研发工程师**：查看和更新任务、提交代码、参与代码审查
- **测试工程师**：创建测试用例、提交缺陷、验证修复
- **产品经理**：管理需求、排定优先级、查看进度报告

## 部署方式

平台支持以下部署方式：



问题：萤火平台支持哪些部署方式？
请在相关句后标注 [来源 n]。


第一版已经出现完整 RAG 闭环：文档变成 Chunk，问题变成关键词，BM25 返回证据，证据变成带引用的 Context。即使暂时不调用 LLM，也能检查最关键的一步：正确证据是否进入 Prompt。

现在故意换一种说法：文档写“部署方式”，用户问“交付形态”。两者意思接近，但没有共同关键词。


In [6]:
same_intent_query = "萤火平台有哪些交付形态？"
failure_results = keyword_search(same_intent_query)
print("查询分词：", tokenize(same_intent_query))
for rank, item in enumerate(failure_results, 1):
    print(f"#{rank} score={item['score']:.3f} {item['chunk']['chunk_id']}")

print("观察：最高分很低或结果依赖偶然的通用词，说明关键词检索无法稳定处理同义表达。")


查询分词： ['萤火', '平台', '有', '哪些', '交付', '形态', '？']
#1 score=3.094 doc-004-incident-handbook:chunk_0
#2 score=2.508 doc-001-product-overview:chunk_0
#3 score=2.046 doc-001-product-overview:chunk_1
观察：最高分很低或结果依赖偶然的通用词，说明关键词检索无法稳定处理同义表达。


## 4 用 Embedding 处理语义相似

Embedding 模型把文本映射为定长向量。训练目标使语义相关文本的向量更接近。生成模型输出文本；Embedding 模型输出向量，两者职责不同。

最常见的比较方式是余弦相似度：

$$
\cos(\theta)=\frac{q\cdot d}{\|q\|_2\|d\|_2}
$$

$q\cdot d$ 是对应维度乘积之和；分母消除向量长度影响。若查询向量和文档向量都已归一化为长度 1，公式就简化为点积 $q\cdot d$。先用三维手工向量观察公式，再接入批量文本表示。


In [7]:
import numpy as np

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 构造 5 个手工向量：模拟 Embedding 空间中的文档和查询
doc_vecs = {
    "D1: 产品说明": np.array([0.8, 0.1, 0.1]),
    "D2: API 文档": np.array([0.1, 0.9, 0.2]),
    "D3: 发布记录": np.array([0.2, 0.1, 0.7]),
    "D4: 故障手册": np.array([0.1, 0.2, 0.9]),
    "D5: 团队规范": np.array([0.5, 0.4, 0.2]),
}
query_vecs = {
    "Q1: 部署方式": np.array([0.7, 0.2, 0.1]),   # 接近 D1（产品说明）
    "Q2: 速率限制": np.array([0.2, 0.8, 0.1]),   # 接近 D2（API 文档）
}

# 计算每个查询与所有文档的相似度
for q_name, q_vec in query_vecs.items():
    print(f"\n{q_name}:")
    scores = [(d_name, cosine_similarity(q_vec, d_vec)) for d_name, d_vec in doc_vecs.items()]
    scores.sort(key=lambda x: x[1], reverse=True)
    for d_name, score in scores:
        marker = " <--" if score == scores[0][1] else ""
        print(f"  vs {d_name}: cos={score:.3f}{marker}")

# 验证：D1 和 D2 之间的相似度（两个不同主题，应较低）
sim_diff = cosine_similarity(doc_vecs["D1: 产品说明"], doc_vecs["D2: API 文档"])
sim_same = cosine_similarity(doc_vecs["D1: 产品说明"], doc_vecs["D1: 产品说明"])
print(f"\nD1 vs D2 (different topics): {sim_diff:.3f}")
print(f"D1 vs D1 (same document): {sim_same:.3f}")
assert sim_diff < sim_same, "Same document should have higher similarity"



Q1: 部署方式:
  vs D1: 产品说明: cos=0.988 <--
  vs D5: 团队规范: cos=0.913
  vs D3: 发布记录: cos=0.426
  vs D2: API 文档: cos=0.396
  vs D4: 故障手册: cos=0.293

Q2: 速率限制:
  vs D2: API 文档: cos=0.987 <--
  vs D5: 团队规范: cos=0.790
  vs D1: 产品说明: cos=0.370
  vs D4: 故障手册: cos=0.351
  vs D3: 发布记录: cos=0.311

D1 vs D2 (different topics): 0.252
D1 vs D1 (same document): 1.000


Q1（部署方式）最接近 D1（产品说明），Q2（速率限制）最接近 D2（API 文档）。这是理想情况——当文档内容和查询意图一致时，向量检索工作得很好。

但这个实验用的是 3 维手工向量。真实 Embedding 的维度由模型决定，语义关系也复杂得多。下一节明确区分离线基线与真实模型实验。


In [8]:
# 本地 Embedding 配置：可以在 .env 中覆盖地址和模型名
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

RUN_LIVE_EMBEDDING = os.getenv("RUN_LIVE_EMBEDDING", "0") == "1"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").rstrip("/")
EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "qwen3-embedding:0.6b")
EMBEDDING_PROVIDER = "ollama" if RUN_LIVE_EMBEDDING else "offline-fixture"
ACTIVE_EMBEDDING_MODEL = EMBEDDING_MODEL if RUN_LIVE_EMBEDDING else "offline-feature-fixture-v1"
QUERY_INSTRUCTION = (
    "Instruct: Given a question about internal product documentation, "
    "retrieve passages that contain evidence for the answer.\nQuery: "
)
print(f"Embedding provider: {EMBEDDING_PROVIDER}; model: {ACTIVE_EMBEDDING_MODEL}")


Embedding provider: offline-fixture; model: offline-feature-fixture-v1


### 4.1 使用 Ollama 运行真实 Embedding

本章的正式实验使用本机 `qwen3-embedding:0.6b` 完成索引和查询。模型下载完成后可以断网运行；文档不会发送到云端。该模型默认输出 1024 维向量，支持中文和检索任务指令。[4]

发布版默认设置 `RUN_LIVE_EMBEDDING=0`，使用固定的离线特征回放验证索引、检索和评测代码，不访问 Ollama。离线回放不能证明真实模型的语义能力；设置 `RUN_LIVE_EMBEDDING=1` 后才会复现本节的真实结论。

读者需要按顺序完成以下准备：

1. 安装 [Ollama](https://ollama.com/download)。
2. 下载模型。只需执行一次：

   ```bash
   ollama pull qwen3-embedding:0.6b
   ```

3. 启动本地服务。如果 Ollama 桌面程序已经运行，不要重复启动：

   ```bash
   ollama serve
   ```

4. 在终端检查模型和服务：

   ```bash
   ollama list
   curl http://127.0.0.1:11434/api/tags
   ```

5. 在 `.env` 中设置 `RUN_LIVE_EMBEDDING=1`，再从头运行 Notebook。此时下面的代码会调用 `POST /api/embed`；如果服务或模型缺失，它会直接给出修复命令，不会静默切回离线回放。[13]

索引时直接编码文档；查询时加入任务指令。Qwen 官方建议为检索查询提供一条英文任务说明，文档本身不加指令。[4]


In [9]:
# 真实路径调用 Ollama；发布验证使用固定离线特征，不发送请求
def ollama_embed(texts: list[str], *, is_query: bool = False) -> np.ndarray:
    """将一批查询或文档转换为真实语义向量。返回 shape=(文本数, 向量维度)。"""
    if not texts:
        return np.empty((0, 0), dtype=np.float32)

    # Qwen3-Embedding 的查询需要任务指令；文档保持原文。
    inputs = [QUERY_INSTRUCTION + text for text in texts] if is_query else texts
    payload = json.dumps({"model": EMBEDDING_MODEL, "input": inputs,
                          "truncate": False}).encode("utf-8")
    request = Request(f"{OLLAMA_BASE_URL}/api/embed", data=payload,
                      headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urlopen(request, timeout=180) as response:
            body = json.loads(response.read().decode("utf-8"))
    except (HTTPError, URLError, TimeoutError) as error:
        raise RuntimeError(
            "无法调用本机 Ollama。请依次执行：\n"
            "  ollama pull qwen3-embedding:0.6b\n"
            "  ollama serve\n"
            "再从头运行 Notebook。"
        ) from error

    vectors = np.asarray(body["embeddings"], dtype=np.float32)
    if vectors.ndim != 2 or vectors.shape[0] != len(texts):
        raise ValueError(f"Ollama 返回形状异常：{vectors.shape}")

    # 逐行 L2 归一化。此后查询向量与文档向量的点积就是余弦相似度。
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("Ollama 返回了零向量，无法计算余弦相似度")
    return vectors / norms

# 离线特征只覆盖固定语料中的五类意图，用于验证数据流，不冒充真实语义模型。
OFFLINE_FEATURE_GROUPS = [
    ("deployment", ("部署", "SaaS", "私有", "混合", "云端", "机房")),
    ("api", ("API", "Token", "速率", "429", "retry_after", "接口")),
    ("release", ("v2.3", "版本", "新增", "附件", "50 MB", "CSV")),
    ("incident", ("数据库", "连接超时", "告警", "消息队列", "Redis", "磁盘")),
    ("convention", ("分支", "commit", "feat-", "fix-", "代码审查", "PR")),
]
OFFLINE_SYNONYMS = {"交付形态": "部署方式", "限流": "速率限制", "提交格式": "commit 格式"}

def offline_feature_embed(texts: list[str], *, is_query: bool = False) -> np.ndarray:
    """为固定教学语料生成 1024 维可复现向量；不用于模型效果结论。"""
    del is_query
    vectors = np.zeros((len(texts), 1024), dtype=np.float32)
    for row, original in enumerate(texts):
        normalized = original
        for source, target in OFFLINE_SYNONYMS.items():
            normalized = normalized.replace(source, target)
        for column, (_, terms) in enumerate(OFFLINE_FEATURE_GROUPS):
            vectors[row, column] = sum(term.lower() in normalized.lower() for term in terms)
        vectors[row, -1] = 0.01  # 未知问题也得到非零向量，便于观察无答案门控。
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

def embed_texts(texts: list[str], *, is_query: bool = False) -> np.ndarray:
    if RUN_LIVE_EMBEDDING:
        return ollama_embed(texts, is_query=is_query)
    return offline_feature_embed(texts, is_query=is_query)

# 先用两句话观察当前路径的输出，避免直接把 Embedding 藏进完整管道。
probe_texts = ["萤火平台支持私有化部署", "今天北京有小雨"]
probe_vectors = embed_texts(probe_texts)
print("输入数量：", len(probe_texts))
print("向量矩阵 shape：", probe_vectors.shape)
print("第一条向量前 8 维：", np.round(probe_vectors[0, :8], 4))
print("每行长度：", np.round(np.linalg.norm(probe_vectors, axis=1), 6))

# 正式索引：同一行的 Chunk 和向量必须始终一一对应。
DOCS_FOR_EMBED = baseline_chunks
texts = [chunk["text"] for chunk in DOCS_FOR_EMBED]
embeddings = embed_texts(texts)
EMBEDDING_DIMENSION = embeddings.shape[1]
print(f"索引：chunks={len(texts)}, model={ACTIVE_EMBEDDING_MODEL}, shape={embeddings.shape}")
assert EMBEDDING_DIMENSION == 1024
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5)


输入数量： 2
向量矩阵 shape： (2, 1024)
第一条向量前 8 维： [1. 0. 0. 0. 0. 0. 0. 0.]
每行长度： [1. 1.]
索引：chunks=25, model=offline-feature-fixture-v1, shape=(25, 1024)


In [10]:
QUESTIONS = [("Q1", "萤火平台支持哪些部署方式？"), ("Q2", "API 的速率限制是多少？"),
    ("Q3", "v2.3 版本新增了什么功能，附件上传限制改成了多少？"),
    ("Q4", "数据库连接超时和消息队列堆积的告警阈值是多少？"),
    ("Q5", "分支命名规则是什么，commit 格式有什么要求？"), ("Q8", "平台的定价方案是什么？")]

def embed_queries(queries: list[str]) -> np.ndarray:
    """查询和文档统一走当前 Embedding 路径。"""
    return embed_texts(queries, is_query=True)

def dense_search(query: str, top_k: int = 3):
    query_vector = embed_queries([query])[0]
    scores = embeddings @ query_vector
    ranked_ids = np.argsort(scores)[::-1][:top_k]
    return [(DOCS_FOR_EMBED[i], float(scores[i])) for i in ranked_ids]

expected = {"Q1":"doc-001", "Q2":"doc-002", "Q3":"doc-003",
            "Q4":"doc-004", "Q5":"doc-005", "Q8":None}
for qid, question in QUESTIONS:
    results = dense_search(question)
    print(qid, [(item[0]["doc_id"], round(item[1], 3)) for item in results],
          "expected=", expected[qid])

# 回到关键词检索失败的同义表达，检查当前路径是否找回产品说明。
synonym_results = dense_search("萤火平台有哪些交付形态？")
print("同义表达 Top-1：", synonym_results[0][0]["doc_id"],
      round(synonym_results[0][1], 3))
assert synonym_results[0][0]["doc_id"].startswith("doc-001")


Q1 [('doc-001-product-overview', 0.986), ('doc-001-product-overview', 0.707), ('doc-001-product-overview', 0.01)] expected= doc-001
Q2 [('doc-002-api-reference', 1.0), ('doc-002-api-reference', 1.0), ('doc-002-api-reference', 1.0)] expected= doc-002
Q3 [('doc-003-changelog', 1.0), ('doc-003-changelog', 0.981), ('doc-005-team-conventions', 0.707)] expected= doc-003
Q4 [('doc-004-incident-handbook', 1.0), ('doc-004-incident-handbook', 1.0), ('doc-004-incident-handbook', 1.0)] expected= doc-004
Q5 [('doc-005-team-conventions', 1.0), ('doc-002-api-reference', 1.0), ('doc-002-api-reference', 1.0)] expected= doc-005
Q8 [('doc-001-product-overview', 1.0), ('doc-002-api-reference', 1.0), ('doc-003-changelog', 0.01)] expected= None
同义表达 Top-1： doc-001-product-overview 0.986


设置 `RUN_LIVE_EMBEDDING=1` 的真实实验曾验证：用户问“交付形态”时，Qwen3-Embedding 能把写着“部署方式”的产品说明排到第一名。发布版保存的离线输出只证明检索管道和断言可重复执行，不作为模型效果证据。真实结论只适用于本章固定语料、模型和查询指令；更换模型、指令、Chunk 策略或业务语料后必须重新评测。

### 4.2 生成模型与 Embedding 模型的区别

生成模型输出文本；Embedding 模型输出可比较的定长向量。生产索引必须记录服务、模型、维度、归一化方式和查询指令版本。本章记录 `ollama`、`qwen3-embedding:0.6b`、实际维度和 `internal-doc-retrieval-v1`。模型升级时构建新索引 generation，不能混用新旧向量。


## 5 从向量数组到检索索引

查询向量生成后，要与所有 Chunk 向量比较并取 Top-K。下面的 NumPy 索引展示最小实现；它保存向量与 Metadata，并执行归一化、点积、过滤和排序。


In [11]:
class MinimalVectorIndex:
    def __init__(self, vectors, metadata):
        self.vectors = vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
        self.metadata = metadata
    def search(self, query_vec, top_k=3, metadata_filter=None):
        scores, results = self.vectors @ (query_vec / max(np.linalg.norm(query_vec), 1e-12)), []
        for i in np.argsort(scores)[::-1]:
            meta = self.metadata[i]
            if metadata_filter and not all(meta.get(k) == v for k, v in metadata_filter.items()): continue
            results.append({**meta, "score": float(scores[i])})
            if len(results) == top_k: break
        return results

index = MinimalVectorIndex(embeddings, DOCS_FOR_EMBED)
q = embed_queries(["数据库连接超时的阈值是多少？"])[0]
public_only = index.search(q, 3, {"access_level": "public"})
print([r["doc_id"] for r in public_only])
assert all(r["access_level"] == "public" and r["doc_id"] != "doc-004-incident-handbook" for r in public_only)
print("OK: metadata filter was applied")


['doc-005-team-conventions', 'doc-005-team-conventions', 'doc-001-product-overview']
OK: metadata filter was applied


这个 20 行的检索器包含了向量数据库的核心逻辑：向量归一化、点积相似度、排序、Top-K 截断和 Metadata 过滤。生产系统在这个基础上增加了 ANN 索引、持久化和并发控制。


### 5.1 精确检索、ANN 与 HNSW

精确检索逐个比较全部向量，结果准确但计算量随向量数增长。ANN 用索引结构减少候选比较，以可测量的召回损失换取延迟和吞吐量；实际分界取决于维度、硬件、过滤条件与 QPS，不能只按“百万向量”判断。

HNSW 用多层近邻图搜索候选。三个常见参数构成明确取舍：

| 参数 | 调大后的主要影响 |
|---|---|
| `M` | 图更大，构建与内存成本增加，召回通常提高 |
| `ef_construction` | 构建更慢，索引质量通常提高 |
| `ef_search` | 查询访问更多节点，召回提高，延迟也提高 |

选型和调参必须在自己的过滤比例、并发与评测集上同时报告 Recall@K、P95/P99 和资源占用。


### 5.2 检索方案能力对比

下表描述能力边界，不用未经基准验证的数据量阈值代替选型。

| 方案 | 部署与持久化 | Metadata 过滤 | 关键词/混合检索 | HA 与扩容 | 适合 | 主要代价 |
|---|---|---|---|---|---|---|
| NumPy | 进程内；自行保存 | 自写 | 自写 | 无 | 教学、精确基线 | 不适合在线多副本 |
| FAISS | 库；索引文件自管 | 需在应用层组合 | 需自建 | 需自行服务化 | 离线评测、定制 ANN | 数据与服务治理自负 |
| Chroma | 本地、服务或云 | 支持 | 支持全文检索，混合能力依版本/部署核验 | 分布式/云能力需按版本核验 | 原型和轻量应用 | 生产边界不能由本地模式外推 |
| pgvector | PostgreSQL 扩展 | SQL 完整 | 与 PostgreSQL FTS 融合 | 复用 PostgreSQL | 已有 PG、事务与 ACL 重要 | ANN 调优和扩容受 PG 架构约束 |
| Qdrant | 服务或云 | 强 | dense+sparse/融合查询 | 分片、复制 | 过滤复杂的向量检索 | 新增独立运维面 |
| Milvus | standalone、分布式或云 | 强 | dense+sparse、BM25/混合 | 分布式扩容 | 大规模、多索引生产 | 组件和运维复杂度较高 |
| Elasticsearch/OpenSearch | 集群或云 | 强 | BM25 与向量原生组合 | 成熟集群能力 | 已有搜索栈、关键词很重要 | 资源与集群调优成本 |
| 托管向量/文件检索 | 云 API | 厂商决定 | 厂商决定 | 厂商负责 | 团队不想维护检索栈 | 数据边界、锁定、费用和可观测性 |

### 5.3 从约束推导选型

先收集 Chunk 数与增长率、读写 QPS、P99、更新时效、过滤选择性、ACL、混合检索需求、可用区与恢复目标，再做同一评测集的候选压测。已有 PostgreSQL 且规模和延迟验证通过时优先评估 pgvector；已有搜索集群且 BM25 重要时优先评估 ES/OpenSearch；需要专用向量过滤与水平扩展时评估 Qdrant/Milvus；原型可用 NumPy、FAISS 或 Chroma。最终结论是压测记录，不是产品名称。


## 6 Hybrid：让语义与精确词项互补

Dense 检索擅长语义相似，BM25 擅长错误码、版本号、API 路径和专名。生产系统常并行取两路候选，再用 Reciprocal Rank Fusion（RRF）融合名次：

$$
RRF(d)=\sum_{a\in A}\frac{1}{k+rank_a(d)}
$$

$A$ 是检索器集合，$rank_a(d)$ 是文档 $d$ 在检索器 $a$ 中的名次。RRF 不直接相加余弦分数和 BM25 分数，因为两者量纲不同。一个 Chunk 同时在两路靠前时会累计更高分。


In [12]:
# BM25 实现
from rank_bm25 import BM25Okapi
import jieba

# 中文分词后构建 BM25
def tokenize(text: str) -> list[str]:
    return list(jieba.cut(text))

chunk_texts = [c["text"] for c in DOCS_FOR_EMBED]
tokenized_chunks = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built: {len(tokenized_chunks)} documents")

def bm25_search(query: str, top_k: int = 3) -> list[dict]:
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [{"chunk": DOCS_FOR_EMBED[i], "bm25_score": float(s)} for i, s in ranked[:top_k]]

# 测试 BM25
for q_id, question in QUESTIONS[:3]:
    print(f"\n{q_id}: {question}")
    for rank, r in enumerate(bm25_search(question, 3)):
        c = r["chunk"]
        print(f"  #{rank+1} [{c['doc_id']}] score={r['bm25_score']:.2f}: "
              f"{c['text'][:80].replace(chr(10), ' ')}...")


BM25 index built: 25 documents

Q1: 萤火平台支持哪些部署方式？
  #1 [doc-001-product-overview] score=9.76: **产品经理**：管理需求、排定优先级、查看进度报告  ## 部署方式  平台支持以下部署方式：  1. **SaaS 云端版**：由萤火团队托管，自动升级，无...
  #2 [doc-001-product-overview] score=8.77: # 萤火协作平台产品说明  ## 产品定位  萤火协作平台是一套面向中小型研发团队的协作工具集，提供任务管理、文档共享、即时通讯和自动化流水线四个核心模块。  ...
  #3 [doc-002-api-reference] score=3.30: # 萤火协作平台 API 参考  ## 认证方式  所有 API 请求需要在 Header 中携带 Bearer Token：  ``` Authorizati...

Q2: API 的速率限制是多少？
  #1 [doc-002-api-reference] score=8.07: # 萤火协作平台 API 参考  ## 认证方式  所有 API 请求需要在 Header 中携带 Bearer Token：  ``` Authorizati...
  #2 [doc-003-changelog] score=5.90: 0 条改为 20 条。  ## v2.2.0（2026-04-20）  ### 新增 - 代码审查模块支持行内评论。 - 流水线支持 GitLab CI 集成。...
  #3 [doc-002-api-reference] score=4.61: 429，Response Body 包含 `retry_after` 字段（单位：秒）。 - 批量导入接口不受速率限制。  ## 分页  支持分页的接口统一使用...

Q3: v2.3 版本新增了什么功能，附件上传限制改成了多少？
  #1 [doc-003-changelog] score=13.73: # 萤火协作平台发布记录  ## v2.3.0（2026-06-15）  ### 新增 - 任务模块支持 CSV 批量导入。 - 消息中心新增 @提及提醒。 -...
  #2 [doc-003-changelog]

In [13]:
# Hybrid 检索：真实 Dense 结果 + BM25 结果，再用 RRF 合并名次
def hybrid_search(query: str, top_k: int = 3, candidate_k: int = 10) -> list[dict]:
    # 第 1 路：Ollama 把查询转为真实语义向量；归一化后点积等于余弦相似度。
    query_vector = embed_queries([query])[0]
    dense_scores = embeddings @ query_vector

    # 第 2 路：BM25 根据分词后的精确词项计算相关性。
    bm25_scores = bm25.get_scores(tokenize(query))
    dense_rank = np.argsort(dense_scores)[::-1][:candidate_k]
    bm25_rank = np.argsort(bm25_scores)[::-1][:candidate_k]

    # RRF 只融合名次，避免直接相加两种不同量纲的分数。
    candidate_ids = set(dense_rank) | set(bm25_rank)
    rrf_scores = {int(index): 0.0 for index in candidate_ids}
    for ranking in (dense_rank, bm25_rank):
        for rank, index in enumerate(ranking, start=1):
            rrf_scores[int(index)] += 1.0 / (60 + rank)

    ranked_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    return [{"chunk": DOCS_FOR_EMBED[i], "dense": float(dense_scores[i]),
             "bm25": float(bm25_scores[i]), "rrf": rrf_scores[i]} for i in ranked_ids]

question = "API 的速率限制是多少？"
for rank, result in enumerate(hybrid_search(question, 3), start=1):
    print(f"#{rank} [{result['chunk']['doc_id']}] RRF={result['rrf']:.4f} "
          f"dense={result['dense']:.3f} BM25={result['bm25']:.3f}")


#1 [doc-002-api-reference] RRF=0.0323 dense=1.000 BM25=8.070
#2 [doc-002-api-reference] RRF=0.0320 dense=1.000 BM25=4.611
#3 [doc-003-changelog] RRF=0.0318 dense=0.817 BM25=5.903


### 6.1 为什么融合后还需要 Rerank

初检追求“别漏掉”，因此候选较多；Rerank 对查询与每个候选成对评分，追求“把最有用的放前面”。真实 Cross-Encoder 更准确但更慢。下面的词项覆盖率只用于离线展示第二阶段排序的数据流。


In [14]:
# 离线 Rerank 基线：比较查询词项有多少出现在候选 Chunk 中
STOPWORDS = {"的", "了", "是", "什么", "哪些", "多少", "如何", "平台", "支持", "吗", "？"}

def informative_terms(text: str) -> set[str]:
    """分词后去掉单字和高频虚词，保留能区分问题主题的词项。"""
    return {term.strip() for term in tokenize(text)
            if len(term.strip()) > 1 and term.strip() not in STOPWORDS}

def rerank_candidates(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    query_terms = informative_terms(query)
    rescored = []
    for item in candidates:
        chunk_terms = informative_terms(item["chunk"]["text"])
        matched_terms = query_terms & chunk_terms
        # 覆盖率 = 候选命中的查询词数 / 查询有效词总数。
        coverage = len(matched_terms) / max(len(query_terms), 1)
        rescored.append({**item, "rerank_score": coverage,
                         "query_terms": query_terms, "matched_terms": matched_terms})

    # 覆盖率相同时保留 RRF 的先后关系，形成稳定排序。
    return sorted(rescored, key=lambda x: (x["rerank_score"], x["rrf"]), reverse=True)[:top_k]

query = "v2.3 版本新增了什么功能？"
before = hybrid_search(query, top_k=8)
after = rerank_candidates(query, before, top_k=3)
print("查询有效词：", sorted(informative_terms(query)))
for rank, item in enumerate(after, start=1):
    print(f"#{rank} {item['chunk']['doc_id']} 命中词={sorted(item['matched_terms'])} "
          f"覆盖率={item['rerank_score']:.2f}")


查询有效词： ['v2.3', '功能', '新增', '版本']
#1 doc-003-changelog 命中词=['v2.3', '新增'] 覆盖率=0.50
#2 doc-003-changelog 命中词=['版本'] 覆盖率=0.25
#3 doc-005-team-conventions 命中词=['功能'] 覆盖率=0.25


## 7 检索失败也可能来自 Chunk 边界

即使检索算法正确，答案的两个事实被切到相邻 Chunk 时，任何单块检索都拿不到完整证据。现在回到 `doc-003-changelog.md`，比较固定长度、标题和段落策略。


### 7.1 六种切分策略

| 策略 | 做法 | 优点 | 风险 |
|------|------|------|------|
| 固定字符 | 每 N 字符切分 | 简单 | 字符数不等于 token 数 |
| Token 长度 | 用目标 tokenizer 控制 | Context 预算可计算 | 与模型版本绑定 |
| 标题/段落 | 按结构切分 | 语义较完整 | 超长章节仍需二次切分 |
| Overlap | 相邻块保留重叠 | 降低边界丢失 | 增加冗余 |
| 语义切分 | 用句向量变化找断点 | 更贴近主题 | 成本高，仍需评测 |
| Parent-Child | 小块召回，返回父块 | 兼顾粒度与上下文 | 需维护父子关系和去重 |

下面比较固定字符、标题和段落合并。段落合并是结构切分基线，不冒充语义切分。


### 7.2 Chunk 边界观察实验

**目标**：观察同一文档经过不同策略后，Chunk 数量、长度和答案跨边界情况。

**输入**：`doc-003-changelog.md`。

**观察点**：v2.3.0 的功能与附件限制能否保留在同一 Chunk。这里只检查边界；检索质量在 §14 用固定问题集验证。


In [15]:
# 固定字符、标题和段落合并三类 Chunk 策略
import re

def chunk_fixed_size(text: str, size: int = 256, overlap: int = 50) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text)); chunks.append(text[start:end])
        if end == len(text): break
        start = end - overlap
    return chunks

def chunk_by_headings(text: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?=\n## )", text) if len(s.strip()) > 20]

def chunk_by_paragraphs(text: str, target: int = 200) -> list[str]:
    chunks, current = [], ""
    for paragraph in [p.strip() for p in text.split("\n\n") if p.strip()]:
        candidate = f"{current}\n\n{paragraph}".strip()
        if current and len(candidate) > target:
            chunks.append(current); current = paragraph
        else: current = candidate
    if current: chunks.append(current)
    return chunks

changelog = docs["doc-003-changelog"]["text"]
strategies = {"Fixed-256": lambda t: chunk_fixed_size(t, 256, 50),
              "Fixed-512": lambda t: chunk_fixed_size(t, 512, 100),
              "Headings": chunk_by_headings, "Paragraphs": chunk_by_paragraphs}
for name, strategy in strategies.items():
    chunks = strategy(changelog)
    complete = any("CSV 批量导入" in x and "50 MB" in x for x in chunks)
    print(f"{name:12s} chunks={len(chunks)}, lengths={[len(x) for x in chunks]}, complete_answer={complete}")


Fixed-256    chunks=4, lengths=[256, 256, 256, 76], complete_answer=True
Fixed-512    chunks=2, lengths=[512, 282], complete_answer=True
Headings     chunks=5, lengths=[203, 163, 115, 137, 53], complete_answer=True
Paragraphs   chunks=4, lengths=[186, 194, 138, 169], complete_answer=True


这个实验只验证 Chunk 边界。`complete_answer=True` 表示同一 Chunk 同时保留了问题所需的两个事实。

边界完整仍不等于检索一定命中。最终选择必须让每种策略分别建索引，再用同一问题集比较 Recall@K、MRR、Context token 数和引用正确性。


### 7.3 Chunk 太大了会怎样，太小了会怎样

| 问题 | 原因 | 后果 |
|------|------|------|
| Chunk 太大（> 1000 tokens） | 一个 Chunk 包含多个主题 | 检索命中但 Context 冗余，浪费 token |
| Chunk 太小（< 50 tokens） | 片段缺少上下文 | 检索命中但模型看不懂这段在说什么 |
| 没有 Overlap | 关键信息正好在 Chunk 边界 | 检索漏掉，因为分布在了相邻 Chunk 中 |
| Overlap 太大 | 冗余内容进入 Context | 浪费 token，降低 Context 的有效信息密度 |

**Chunk 大小和 Overlap 是一对矛盾**。本章的选择：固定 256 字符、Overlap 50 字符作为基线，后续通过评测（§14）选择最优策略。


### 7.4 Document ID、Chunk ID 与版本追踪

生产系统应区分稳定身份和派生版本：

```text
document_id = 业务系统稳定主键
source_revision = 文档事实源版本
pipeline_version = parser + chunker + embedding 配置版本
chunk_id = hash(document_id + source_revision + pipeline_version + chunk_index)
content_hash = hash(chunk_text)
```

`document_id` 在修改后保持不变；`chunk_id` 标识某个文档版本下的派生 Chunk。幂等摄取以 `(document_id, source_revision, pipeline_version)` 为任务键。新 generation 构建并校验完成后再原子切换别名，避免先删后插造成服务空窗。

### 7.5 用评测选择 Chunk 策略

固定问题与相关 Chunk 标注，为每种策略建立独立索引，比较 Recall@K、MRR、Context token 数、答案忠实度和引用正确性。每次结果同时记录语料、模型与 Chunk 配置版本。


## 8 Metadata、权限与文档更新

Chunk 除正文外还要携带稳定 ID、来源、版本、权限和切分策略。检索时必须在文本进入 Reranker 和 Context 前应用 ACL；文档修改和删除时用这些 ID 清理旧派生产物。


### 8.1 Metadata 设计

每条 Chunk 必须携带以下 Metadata：

| 字段 | 来源 | 用途 |
|------|------|------|
| `doc_id` | 文档入库时分配 | 引用溯源、删除传播 |
| `chunk_id` | Chunk 策略生成 | 唯一标识、更新定位 |
| `title` | 文档解析提取 | 引用展示 |
| `source_path` | 文件路径 | 调试和溯源 |
| `version` | 文档元数据 | 版本对比 |
| `updated_at` | 文件系统或 CMS | 增量更新判断 |
| `access_level` | 业务系统 | 权限过滤（public / team / ops-only） |
| `tags` | 人工或自动分类 | 类别过滤 |
| `chunk_index` | Chunk 策略生成 | 相邻 Chunk 拼接 |


### 8.2 文档的增删改与索引同步

文档变化时，索引需要同步更新。

| 操作 | 索引动作 |
|------|---------|
| 新增文档 | 解析、Chunk、Embedding、插入索引 |
| 修改文档 | 按 `doc_id` 删除旧 Chunk，重新生成并插入新 Chunk |
| 删除文档 | 按 `doc_id` 删除所有关联 Chunk |
| 权限变更 | 更新 Metadata 中的 `access_level`，不重建 Embedding |

核心原则：每种文档操作都有一组确定性的 Chunk ID 变更。修改文档时，先删后插保证幂等；删除文档时，依赖 Metadata 中的 `doc_id` 做批量清理。


## 9 把改进后的链路组装起来

现在已经有 Chunk、向量表示、BM25、RRF、Rerank 和 ACL。下面把它们收束成两个函数：`build_index` 负责提前建库，`rag_query` 负责每次查询。


In [16]:
# 完整索引管道：源文件 -> Chunk 记录 -> 当前 Embedding 向量矩阵
import hashlib

def build_index(doc_dir: str):
    chunks = []
    for file_path in sorted(Path(doc_dir).glob("*.md")):
        text = file_path.read_text(encoding="utf-8")
        title_match = re.search(r"^#\s+(.+)", text, re.MULTILINE)
        title = title_match.group(1) if title_match else file_path.stem

        for chunk_index, chunk_text in enumerate(chunk_fixed_size(text, 256, 50)):
            content_hash = hashlib.sha256(chunk_text.encode()).hexdigest()
            chunks.append({"doc_id": file_path.stem,
                "chunk_id": f"{file_path.stem}:{chunk_index}:{content_hash[:8]}",
                "title": title, "source_path": str(file_path), "content_hash": content_hash,
                "embedding_provider": EMBEDDING_PROVIDER, "embedding_model": ACTIVE_EMBEDDING_MODEL,
                "embedding_dimension": EMBEDDING_DIMENSION,
                "query_instruction_version": "internal-doc-retrieval-v1",
                "chunk_policy": "fixed-256-overlap-50-v1",
                "access_level": "ops-only" if "ops-only" in text.lower() else "public",
                "text": chunk_text})

    # 文档不加查询指令；返回矩阵第 i 行与 chunks[i] 一一对应。
    vectors = embed_texts([chunk["text"] for chunk in chunks])
    return chunks, vectors

all_chunks, all_vectors = build_index(str(DOCS_DIR))
print("第一条索引记录：", {key: all_chunks[0][key] for key in
      ("chunk_id", "embedding_provider", "embedding_model", "embedding_dimension")})
print(f"索引结果：{len(all_chunks)} 条 Chunk，向量矩阵 shape={all_vectors.shape}")
assert all_vectors.shape == (len(all_chunks), EMBEDDING_DIMENSION)
assert len({chunk['chunk_id'] for chunk in all_chunks}) == len(all_chunks)


第一条索引记录： {'chunk_id': 'doc-001-product-overview:0:b27fc224', 'embedding_provider': 'offline-fixture', 'embedding_model': 'offline-feature-fixture-v1', 'embedding_dimension': 1024}
索引结果：25 条 Chunk，向量矩阵 shape=(25, 1024)


In [17]:
# === OpenAI-compatible DeepSeek API 配置；默认不发起网络请求 ===
from dotenv import find_dotenv, load_dotenv
dotenv_path = find_dotenv(usecwd=True)
if dotenv_path: load_dotenv(dotenv_path, override=False)

RUN_LIVE_LLM = os.getenv("RUN_LIVE_LLM", "0") == "1"
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://api.deepseek.com")
LLM_MODEL = os.getenv("LLM_MODEL", "deepseek-v4-pro")
def llm_ready(): return bool(RUN_LIVE_LLM and DEEPSEEK_API_KEY and LLM_MODEL)
print(f"Live LLM={llm_ready()}, model={LLM_MODEL}")


Live LLM=False, model=deepseek-v4-pro


In [18]:
# 完整查询管道：问题 -> 候选 -> 权限过滤 -> Rerank -> Context -> 答案
def rag_query(question: str, top_k: int = 3, access_level: str = "public") -> dict:
    # 步骤 1：Dense 和 BM25 各自召回，RRF 合并成较大的候选集。
    recalled = hybrid_search(question, top_k=min(10, len(DOCS_FOR_EMBED)))

    # 步骤 2：在候选文本进入 Reranker、Context 和日志前执行权限过滤。
    allowed = [item for item in recalled if item["chunk"]["access_level"] == access_level]

    # 步骤 3：词项覆盖率 Rerank 基线重新打分，只保留最终送入 Context 的 top_k。
    candidates = rerank_candidates(question, allowed, top_k)

    # 步骤 4：把最高词项覆盖率当作证据充分度。本章的 0.5 只适用于固定样例；
    # 生产阈值需要在包含有答案和无答案问题的验证集上校准。
    evidence_score = max((item["rerank_score"] for item in candidates), default=0.0)
    has_answer = evidence_score >= 0.5

    # 步骤 5：只有证据通过门控时才组装 Context。来源编号与候选顺序一致。
    context = "\n\n---\n\n".join(
        f"[来源 {rank}] {item['chunk']['title']} | {item['chunk']['chunk_id']}\n{item['chunk']['text']}"
        for rank, item in enumerate(candidates, start=1)) if has_answer else ""
    prompt = f"""你是文档问答助手。只根据资料回答；资料不足时输出“根据现有资料，无法找到相关信息”。
资料：
{context}
问题：{question}
答案必须在相关句后标注 [来源 n]。"""

    # 步骤 6：拒答无需调用模型；离线模式也不发送网络请求。
    if not has_answer:
        answer = "根据现有资料，无法找到相关信息"
    elif llm_ready():
        from openai import OpenAI
        client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=LLM_BASE_URL)
        response = client.chat.completions.create(model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}], temperature=0)
        answer = response.choices[0].message.content
    else:
        answer = f"[离线模式] 证据通过门控；Top-1 为 [来源 1] {candidates[0]['chunk']['title']}。"

    return {"question": question, "has_answer": has_answer, "evidence_score": evidence_score,
        "candidates": [{"chunk_id": item["chunk"]["chunk_id"],
                        "doc_id": item["chunk"]["doc_id"], "score": item["rerank_score"]}
                       for item in candidates],
        "context_length": len(context), "answer": answer,
        "prompt": None if llm_ready() else prompt}

result = rag_query("萤火平台支持哪些部署方式？")
print("是否可回答：", result["has_answer"], "证据分数：", result["evidence_score"])
print("候选来源：", [item["doc_id"] for item in result["candidates"]])
print("答案：", result["answer"])
assert result["has_answer"] and result["candidates"][0]["doc_id"].startswith("doc-001")


是否可回答： True 证据分数： 1.0
候选来源： ['doc-001-product-overview', 'doc-001-product-overview', 'doc-002-api-reference']
答案： [离线模式] 证据通过门控；Top-1 为 [来源 1] 萤火协作平台产品说明。


### 9.1 引用与无答案

引用不是在答案末尾罗列 URL。系统要保存“答案句 -> `chunk_id` -> 文档版本 -> 原文位置”的映射，并验证被引用片段确实支撑该句。无答案门控应在调用生成模型前执行，避免把无关 Context 交给模型后再期待它自行拒答。下面同时验证缺失知识拒答和 ACL。


In [19]:
result_q8 = rag_query("平台的定价方案是什么？")
print("Q8:", result_q8["answer"], result_q8["evidence_score"])
assert not result_q8["has_answer"]

result_q9 = rag_query("数据库连接超时的阈值是多少？", access_level="public")
assert all(not x["doc_id"].startswith("doc-004") for x in result_q9["candidates"])
print("Q9 public sources:", [x["doc_id"] for x in result_q9["candidates"]])


Q8: 根据现有资料，无法找到相关信息 0.0
Q9 public sources: ['doc-005-team-conventions', 'doc-005-team-conventions', 'doc-003-changelog']


离线模式下 LLM 调用被跳过，但检索、过滤、Context 组装和来源追踪的完整代码路径都已覆盖。设置 `RUN_LIVE_LLM=1` 可启用真实 DeepSeek 生成。


## 10 用固定问题集评测，而不是凭感觉挑结果

Hit Rate@K 检查 Top-K 是否至少命中一个相关 Chunk；Recall@K 检查全部相关 Chunk 找回多少；MRR 关注第一个相关结果的位置；NDCG 支持多级相关性。生成侧还要评测 Faithfulness、Citation Correctness 和拒答。

下面的微型评测只验证本章固定语料的召回、排序和无答案门控。它不代表线上质量。


In [20]:
# 固定评测集：四个可回答问题 + 一个无答案问题
EVAL_QUESTIONS = [
 {"id":"Q1", "question":"萤火平台支持哪些部署方式？", "sources":{"doc-001"}, "has_answer":True},
 {"id":"Q2", "question":"API 的速率限制是多少？", "sources":{"doc-002"}, "has_answer":True},
 {"id":"Q3", "question":"v2.3 版本新增了什么功能？", "sources":{"doc-003"}, "has_answer":True},
 {"id":"Q5", "question":"分支命名规则是什么？", "sources":{"doc-005"}, "has_answer":True},
 {"id":"Q8", "question":"平台的定价方案是什么？", "sources":set(), "has_answer":False},
]

reciprocal_ranks, hit_count = [], 0
for case in EVAL_QUESTIONS:
    result = rag_query(case["question"])
    if case["has_answer"]:
        assert result["has_answer"], f"{case['id']} 有标注答案却被错误拒答"
        # 找出第一个期望来源出现的位置。排名 1、2、3 分别贡献 1、1/2、1/3。
        ranked_doc_ids = [item["doc_id"] for item in result["candidates"]]
        relevant_ranks = [rank for rank, doc_id in enumerate(ranked_doc_ids, start=1)
                          if any(doc_id.startswith(prefix) for prefix in case["sources"])]
        hit_count += bool(relevant_ranks)
        reciprocal_ranks.append(1 / relevant_ranks[0] if relevant_ranks else 0)
    else:
        # 无答案样例单独验证拒答，避免它被排除在质量指标之外。
        assert not result["has_answer"]

answerable_count = sum(case["has_answer"] for case in EVAL_QUESTIONS)
metrics = {"Hit Rate@3": hit_count / answerable_count,
           "MRR": sum(reciprocal_ranks) / answerable_count}
print(metrics)
assert metrics["Hit Rate@3"] == 1.0 and metrics["MRR"] >= 0.75


{'Hit Rate@3': 1.0, 'MRR': 1.0}


### 10.1 错误定位

检索或生成出错时，按以下顺序排查：

1. 文档解析是否正确（表格和代码块是否完整）。
2. Chunk 是否包含了关键信息（Chunk 边界是否切断了答案）。
3. Embedding 模型是否适合当前语言和领域。
4. 检索 Top-K 是否包含了正确答案的 Chunk。
5. Rerank 是否把正确答案排出了 Top-N。
6. LLM 是否忽略了 Context 中的正确信息（编造或漏读）。


## 11 工程产物、事实源与持久化

| 产物 | 分类 | 版本/审计键 | 丢失后的动作 |
|---|---|---|---|
| 原始文档及业务 Metadata/ACL | 事实源 | `document_id`、业务版本 | 从源系统恢复 |
| 解析结果、Chunk、Embedding | 可重建派生产物 | parser/chunk/model revision + content hash | 按相同配置重建 |
| 向量索引、关键词索引 | 服务索引 | index generation | 从派生产物重建 |
| 查询/Embedding 缓存 | 缓存 | TTL、模型版本 | 丢弃或回填 |
| 配置与模型版本 | 需版本管理 | immutable revision | 从配置库恢复 |
| 评测集 | 需版本管理的数据 | dataset revision | 从评测仓恢复 |
| 检索日志与引用记录 | 审计数据 | request/tenant/user/index generation | 按合规策略归档 |

原始文件、业务 Metadata 和 ACL 的事实源归属要在系统设计中明确。Chunk 不是事实源；即使可重建，也应保留生成配置，因为重建成本可能很高。日志至少记录查询版本、过滤条件、各路候选、Rerank 分数、最终引用与索引 generation，并对敏感文本脱敏。容量应由实际 `N × dimension × dtype`、图索引开销、副本数和增长率计算。


## 12 生产架构与索引生命周期

```mermaid
flowchart LR
  S["事实源 + 变更日志"] --> I["幂等摄取"] --> P["解析与 Chunk"] --> E["Embedding"]
  P --> K["关键词索引 generation B"]
  E --> V["向量索引 generation B"]
  I --> D["检查点 / DLQ / 对账"]
  Q["在线查询 + 身份"] --> A["ACL 过滤"] --> R["Hybrid + Rerank"] --> C["Context + 引用"] --> G["生成/拒答"]
  V --> R
  K --> R
```

在线查询与离线摄取独立扩容。全量建库读取一致快照；增量链路消费带版本的新增、修改和删除事件。稳定 `document_id` 由业务身份产生，`chunk_id` 由 document/version/policy/position 或内容散列确定；每次 upsert/delete 带 generation，重复与乱序事件不会复活旧 Chunk。

Embedding 模型、解析器或 Chunk 策略升级时构建完整的新 generation，同时写向量与关键词索引。对账确认文档数、Chunk 数、删除墓碑和评测指标后，原子切换读别名；观察期内可回滚，之后再回收旧索引。索引与事实源不一致时停增量、从检查点重放；无法证明一致性时从事实源快照全量重建。

多租户先确定隔离级别：独立实例/collection 强隔离，或共享索引加 `tenant_id` 与 ACL 预过滤。授权条件必须进入召回查询，不能等 Context 组装后再删。备份事实源、配置、评测集和审计记录，并演练恢复；索引是否备份取决于重建时间能否满足 RTO。明确 RPO、RTO、保留期与密钥轮换。

容量和成本用工作负载推导：原始向量约为 `N × dimension × bytes × replicas`，另计图/倒排索引、Metadata、增长余量；吞吐要分别压测 embedding、过滤后 ANN、BM25、Rerank 和 LLM。SLO 记录 P50/P95/P99、QPS、错误率、召回质量和单查询成本，不预设通用的 500 ms 或固定月费。


## 13 部署与托管形态

进程内库适合原型和离线评测；独立检索服务适合需要独立扩缩容、版本切换和统一 ACL 的生产系统；Sidecar 适合已有 Kubernetes 且租户/版本与 Pod 生命周期一致的场景；MCP 只标准化工具暴露，不替代索引链路。

Managed File Search 通常打包上传、解析、切分、索引和检索，但各厂商在可配置 Chunk、过滤、混合检索、数据驻留、删除时效、日志和计费上不同。选用前用合同与官方文档核验这些边界，并准备数据导出和替代路径。

| 约束 | 优先评估 |
|---|---|
| 最快验证、无运维团队 | 托管 File Search 或托管向量服务 |
| 严格数据驻留/私网 | 自建独立服务或符合区域要求的托管服务 |
| 多业务共享、需要蓝绿索引 | 独立 RAG Service |
| Agent 需要发现检索能力 | 在 RAG Service 外增加 MCP Tool |
| 单机离线实验 | 进程内 NumPy/FAISS/Chroma |

最终选择仍需代入数据量、QPS、P99、更新频率、ACL、现有基础设施和团队值班能力。


## 14 安全、隐私与故障

检索系统有两类注入：用户问题中的直接注入，以及被索引文档中的间接注入。检索内容是不可信数据，必须与系统指令分隔；生成模型不得因文档文本而调用工具、泄露系统提示或改变授权。入库前做恶意内容与敏感信息扫描，输出做 DLP/密钥检测。

越权检索要在召回前用可信身份注入 `tenant_id`、主体、资源和策略版本；不能接受模型或用户自己提供的权限字段。缓存键包含租户和授权版本，日志脱敏，引用链接再次鉴权。用 canary 文档和跨租户测试持续验证没有敏感泄漏。

| 现象 | 首要证据 | 恢复动作 |
|---|---|---|
| 删除后仍命中 | 事件 offset、墓碑、两类索引 generation | 重放删除并对账 |
| 向量维度不符 | 模型 revision 与 schema | 阻止混写，切回旧 generation |
| 召回下降 | 固定评测集分路指标 | 回滚模型/Chunk/索引版本 |
| P99 突增 | 过滤选择性、`ef_search`、Rerank 批次 | 降级候选数或扩容 |
| 引用不支撑答案 | claim—citation 审计记录 | 拒答或重新生成 |

成本占比没有通用比例。分别度量每百万摄取 token、每次查询的 embedding/检索/rerank/生成成本，以及存储、副本、跨区流量和运维人力。


## 15 RAG 的扩展与边界

长上下文降低了小型、静态语料搭建索引的必要性，但输入 token 增多会影响费用和延迟，模型也可能忽略长输入中间的信息。直接输入仍可记录所用文档与位置，因此“不可追溯”不是必然；RAG 的优势在于选择证据、更新、权限过滤和大规模语料成本。两种方案要在同一问题集上比较质量、延迟和费用。[9]

Agentic RAG 让模型规划、改写并多次检索，适合需要探索或交叉验证的问题，但增加延迟、成本与不可预测路径；普通单轮检索仍是可控基线。GraphRAG 显式构建实体、关系和社区摘要，适合全局归纳或多跳问题，代价是抽取、更新和评测链路更复杂。[10] 多模态 RAG 则要同时治理 OCR、表格、图片位置和跨模态引用。

Managed File Search 是交付形态，不是新的检索原理。产品文档只能证明厂商承诺的当前能力，不能证明它一定优于自建系统。


## 16 与产研协作助手的关系

第 11 章的产研协作助手需要检索以下文档类型：

- 需求文档和技术方案
- API 规范和接口文档
- 项目发布记录和变更日志
- 测试报告和缺陷统计

本章的 RAG 系统可以为产研协作助手提供文档检索和引用能力。具体接入方案在第 11 章展开。

本章不实现产研协作助手业务流程。


## 17 本章总结

本章沿一条问答主线完成了以下演进：

```text
关键词 RAG 闭环
  -> 同义表达失败
  -> Embedding 与余弦相似度
  -> Dense + BM25 + RRF
  -> Rerank
  -> Chunk 边界修复
  -> Metadata / ACL / 无答案
  -> 固定评测集
  -> 版本化、可恢复的生产索引
```

上线前必须回答：事实源在哪里；文档增删改怎样传播；使用哪个模型和 Chunk 策略版本；ACL 在哪一步生效；拒答阈值如何校准；索引如何重建与回滚；质量、延迟和成本怎样共同验收。


## 参考文献

[1] LEWIS P, PEREZ E, PIKTUS A, et al. Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks[C]//Advances in Neural Information Processing Systems. 2020.

[2] MALKOV Y A, YASHUNIN D A. Efficient and Robust Approximate Nearest Neighbor Search Using Hierarchical Navigable Small World Graphs[J]. IEEE Transactions on Pattern Analysis and Machine Intelligence, 2020, 42(4): 824-836.

[3] ROBERTSON S, ZARAGOZA H. The Probabilistic Relevance Framework: BM25 and Beyond[J]. Foundations and Trends in Information Retrieval, 2009, 3(4): 333-389.

[4] QWEN TEAM. Qwen3 Embedding[EB/OL]. [2026-07-14]. https://github.com/QwenLM/Qwen3-Embedding.

[5] PGVECTOR. pgvector: Open-source vector similarity search for Postgres[EB/OL]. [2026-07-14]. https://github.com/pgvector/pgvector.

[6] QDRANT. Hybrid Queries[EB/OL]. [2026-07-14]. https://qdrant.tech/documentation/concepts/hybrid-queries/.

[7] MILVUS. Multi-Vector Hybrid Search[EB/OL]. [2026-07-14]. https://milvus.io/docs/multi-vector-search.md.

[8] ELASTIC. Hybrid search[EB/OL]. [2026-07-14]. https://www.elastic.co/docs/solutions/search/hybrid-search.

[9] LIU N F, LIN K, HEWITT J, et al. Lost in the Middle: How Language Models Use Long Contexts[J]. Transactions of the Association for Computational Linguistics, 2024, 12: 157-173.

[10] EDGE D, TRINH H, CHENG N, et al. From Local to Global: A Graph RAG Approach to Query-Focused Summarization[EB/OL]. arXiv:2404.16130, 2024[2026-07-14]. https://arxiv.org/abs/2404.16130.

[11] DEEPSEEK. Your First API Call[EB/OL]. [2026-07-14]. https://api-docs.deepseek.com/.

[12] OPENAI. File search[EB/OL]. [2026-07-14]. https://platform.openai.com/docs/guides/tools-file-search.

[13] OLLAMA. Generate embeddings[EB/OL]. [2026-07-14]. https://docs.ollama.com/api/embed.

[13] OLLAMA. Generate embeddings[EB/OL]. [2026-07-14]. https://docs.ollama.com/api/embed.
